# 01 Build Master Table

**Purpose:** Merge the 7 OULAD tables into a single master table with:
- Binary target variable (`at_risk`)
- Three feature groups: Demographics, Engagement, Performance
- Temporal column for checkpoint slicing

**Output:** `data/interim/master_raw.parquet`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RAW_DIR = Path('../data/raw')
INTERIM_DIR = Path('../data/interim')
INTERIM_DIR.mkdir(exist_ok=True, parents=True)

## 1. Load Raw Data

In [ ]:
assessments = pd.read_csv(RAW_DIR / 'assessments.csv')
courses = pd.read_csv(RAW_DIR / 'courses.csv')
student_assessment = pd.read_csv(RAW_DIR / 'studentAssessment.csv')
student_info = pd.read_csv(RAW_DIR / 'studentInfo.csv')
student_registration = pd.read_csv(RAW_DIR / 'studentRegistration.csv')
student_vle = pd.read_csv(RAW_DIR / 'studentVle.csv')
vle = pd.read_csv(RAW_DIR / 'vle.csv')

print(f"Students: {student_info.shape[0]:,}")
print(f"VLE interactions: {student_vle.shape[0]:,}")
print(f"Assessment submissions: {student_assessment.shape[0]:,}")

## 2. Create Binary Target Variable

- **at_risk = 1**: Fail or Withdrawn
- **at_risk = 0**: Pass or Distinction

In [ ]:
student_info['at_risk'] = student_info['final_result'].isin(['Fail', 'Withdrawn']).astype(int)

print("\nTarget distribution:")
print(student_info['at_risk'].value_counts())
print(f"At-risk rate: {student_info['at_risk'].mean():.1%}")

## 3. Aggregate Engagement Features (VLE)

Per student-module-presentation:
- Total clicks
- Days active
- Activity by resource type

In [ ]:
# Join VLE activity with resource metadata
vle_full = student_vle.merge(vle, on=['id_site', 'code_module', 'code_presentation'], how='left')

# Aggregate per student
engagement = vle_full.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    total_clicks=('sum_click', 'sum'),
    days_active=('date', 'nunique'),
    num_resources=('id_site', 'nunique')
).reset_index()

# Activity by resource type (pivot)
activity_by_type = vle_full.groupby(
    ['code_module', 'code_presentation', 'id_student', 'activity_type']
)['sum_click'].sum().unstack(fill_value=0)
activity_by_type.columns = [f'clicks_{col}' for col in activity_by_type.columns]
activity_by_type = activity_by_type.reset_index()

engagement = engagement.merge(activity_by_type, on=['code_module', 'code_presentation', 'id_student'], how='left')

print(f"\nEngagement features: {engagement.shape}")

## 4. Aggregate Performance Features (Assessments)

Per student:
- Mean score
- Number of submissions
- Late submission rate

In [ ]:
# Join assessment submissions with metadata (merge on id_assessment only)
assess_full = student_assessment.merge(
    assessments, on='id_assessment', how='left'
)

# Late submission flag
assess_full['is_late'] = (assess_full['date_submitted'] > assess_full['date']).astype(int)

# Aggregate per student
performance = assess_full.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    mean_score=('score', 'mean'),
    num_submissions=('id_assessment', 'count'),
    late_rate=('is_late', 'mean')
).reset_index()

print(f"\nPerformance features: {performance.shape}")

## 5. Build Master Table

Join demographics (student_info) with engagement and performance features.

In [ ]:
master = student_info.copy()

# Merge engagement
master = master.merge(
    engagement,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# Merge performance
master = master.merge(
    performance,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# Fill missing engagement/performance with 0 (no activity)
engagement_cols = engagement.columns.difference(['code_module', 'code_presentation', 'id_student'])
performance_cols = performance.columns.difference(['code_module', 'code_presentation', 'id_student'])
master[engagement_cols] = master[engagement_cols].fillna(0)
master[performance_cols] = master[performance_cols].fillna(0)

print(f"\nMaster table: {master.shape}")
print(f"Features: {master.shape[1]}")
print(f"\nMissing values:")
print(master.isnull().sum()[master.isnull().sum() > 0])

## 6. Add Temporal Information

Join course length for checkpoint slicing later.

In [ ]:
master = master.merge(
    courses[['code_module', 'code_presentation', 'module_presentation_length']],
    on=['code_module', 'code_presentation'],
    how='left'
)

print(f"\nCourse length distribution:")
print(master['module_presentation_length'].describe())

## 7. Save Master Table

In [ ]:
output_path = INTERIM_DIR / 'master_raw.parquet'
master.to_parquet(output_path, index=False)

print(f"\n✓ Master table saved to {output_path}")
print(f"  Shape: {master.shape}")
print(f"  Size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")

## 8. Summary Statistics

In [ ]:
print("\n=== MASTER TABLE SUMMARY ===")
print(f"Total records: {len(master):,}")
print(f"\nFeature groups:")
print(f"  Demographics: {len([c for c in master.columns if c in student_info.columns])}")
print(f"  Engagement: {len(engagement_cols)}")
print(f"  Performance: {len(performance_cols)}")
print(f"\nTarget balance:")
print(master['at_risk'].value_counts(normalize=True))
print(f"\nSample preview:")
print(master.head(3))